根据对话记录，以下是实验验证情况的简要总结：

---

## 实验验证情况

### 干净条件下的比较（Section 5.2）

| 实验 | 脚本/位置 | 数据集 | 模型 | 结果 |
|------|----------|--------|------|------|
| **Experiment 1** | `run_experiment1_unified.py` | dev-clean-2 (26 spk) | mel_mlp, mfcc_mlp | 94-99% |
| **混沌模型训练** | `train_chaotic.py` | dev-clean-2 (26 spk) | C-HiLAP | 59.32-69.49% |
| **Bifurcation实验** | 未明确 | train-clean-100 (100 spk) | C-HiLAP + bifurcation | **97.46%** |

⚠️ **问题**：5.2使用的97.46%数据来自100 speakers + bifurcation control实验，但该实验的具体代码/日志**未在对话中明确记录**。

---

### 噪声环境下的比较（Section 5.3）

| 实验 | 脚本 | 数据集 | 测试模型 | 状态 |
|------|------|--------|---------|------|
| **Experiment 2** | `run_experiment2_simple.py` | dev-clean-2 (26 spk) | mel_mlp, mfcc_mlp, chaotic | ✅ 已完成 |

**关键结果**（在26 speakers, dev-clean-2上）：
- C-HiLAP clean: 59.32% → 20dB SNR保持率 **97.5%**
- Baseline clean: 94-99% → 20dB SNR保持率 **45-47%**

---

## ⚠️ 数据一致性问题

| Section | 使用的数据 | 来源 | 是否一致 |
|---------|-----------|------|---------|
| 5.2.1 C-HiLAP: 97.46% | 100 spk + bifurcation | 不明确 | ❓ |
| 5.2.2 Baselines: 94-95% | 26 spk (推测也有100 spk版本) | Exp 1 | ⚠️ |
| 5.3 噪声测试 | 26 spk, C-HiLAP: 59.32% | Exp 2 | ✅ |

**核心问题**：5.2和5.3使用的C-HiLAP准确率不同（97.46% vs 59.32%），因为来自不同实验配置。

---

## 建议

1. **确认97.46%的实验日志**：找到对应的训练日志/checkpoint
2. **统一实验条件**：理想情况下，5.2和5.3应使用同一模型
3. **或在论文中说明**：5.2用100 spk主实验，5.3用26 spk消融实验的模型

**Experiment 1 (实验一)** 的核心目的是建立 **传统基准 (Traditional Baselines)**。

简单来说，它的作用是回答这个问题：**“如果不使用任何复杂的混沌理论（Chaotic Theory）或高级架构，仅使用最标准的音频特征和最简单的神经网络，我们能达到什么样的声纹识别准确率？”**

以下是 Experiment 1 具体验证的详细内容：

### 1. 验证传统声学特征的有效性 (Feature Effectiveness)

实验通过对比两种最经典的音频特征，验证它们在当前数据集上的表现差异：

* **Mel Spectrogram (梅尔频谱图):** 包含更丰富的频谱细节（代码中默认 `n_mels=80`）。通常被认为包含更多信息，但也包含更多噪声。
* **MFCC (梅尔频率倒谱系数):** 它是高度压缩的特征（代码中默认 `n_mfcc=40` 或 `13`），去除了许多冗余信息，更接近人类听觉感知。
* **验证点：** 在相同的模型结构下，到底是“信息量大”的 Mel 频谱好，还是“去相关性强”的 MFCC 好？这为后续混沌特征的性能提供了参考坐标。

### 2. 确立性能的“下限” (Establishing a Baseline)

* **模型架构：** 使用了 `TraditionalMLPBaseline`，这是一个非常基础的多层感知机（MLP）。
* 结构：`Input -> Linear -> BatchNorm -> ReLU -> Dropout -> Output`。
* 特点：它没有时间序列处理能力（没有 LSTM/GRU），没有注意力机制，也没有混沌动力学。它只是简单地对特征取平均（Mean Pooling）后进行分类。


* **验证点：** 这个实验确立了**最低合格线**。如果后续开发的“混沌混合模型”或“复杂深度学习模型”的准确率不能显著高于 Experiment 1 的结果，那么新模型就没有存在的价值。

### 3. 验证统一训练流程的公平性 (Unified Training Pipeline)

`run_experiment1.py` 强调了 "Unified Training Script"（统一训练脚本），这是为了控制变量：

* **控制变量：** 所有模型使用完全相同的：
* 数据加载器 (`create_speaker_dataloaders`)
* 优化器 (`AdamW`)
* 学习率调度 (`ReduceLROnPlateau`)
* Batch Size (32) 和 Epochs (100)
* 随机种子 (Seed 42)


* **验证点：** 确保未来实验中性能的提升是源于**模型架构的改进**（例如引入混沌特征），而不是因为调参技巧、数据切分不同或训练轮数不同带来的运气。

### 4. 系统完整性检查 (Sanity Check)

在运行复杂的实验之前，Experiment 1 充当了“系统健康检查”的角色：

* 验证数据读取路径是否正确。
* 验证 `TraditionalFeaturePipeline` 是否能正常提取特征。
* 验证 GPU/CPU 调度、日志记录 (`logging`) 和 检查点保存 (`checkpoints`) 功能是否正常工作。

### 总结

**Experiment 1 是整个项目的地基。**

| 实验对象 | 验证内容 | 预期结果 (假设) |
| --- | --- | --- |
| **mel_mlp** | 高维特征 + 简单模型 | 准确率较高，计算量稍大，作为强基准。 |
| **mfcc_mlp** | 低维特征 + 简单模型 | 准确率可能略低或持平，但计算速度快，作为轻量级基准。 |

**对于你的论文或报告，你可以这样描述 Experiment 1：**

> "Experiment 1 旨在通过标准的 MLP 网络评估 Mel 频谱图与 MFCC 特征在受控环境下的基准性能，从而为评估后续提出的混沌特征提取算法的有效性提供严格的对比基线（Baseline）。"

In [17]:
!python train_baselines.py --method mel_mlp --epochs 100 --batch_size 32

✓ Added paths to sys.path: /scratch/project_2003370/yueyao/Model
✓ Traditional features imported successfully
Fallback import setup: added /scratch/project_2003370/yueyao/Model to path

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0

# 26 speakers

In [2]:
!python run_experiment1.py --model mel_mlp

  ✓ Dataset loader imported successfully
✓ Data package initialized (circular import free)

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(

In [3]:
!python run_experiment1.py --model mfcc_mlp

  ✓ Dataset loader imported successfully
✓ Data package initialized (circular import free)

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(

# 251 speakers

In [6]:
!python run_experiment1.py --model mel_mlp --data_dir /scratch/project_2003370/yueyao/dataset/train-clean-100/LibriSpeech/train-clean-100

  ✓ Dataset loader imported successfully
✓ Data package initialized (circular import free)

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(

In [1]:
!python run_experiment1.py --model mfcc_mlp --data_dir /scratch/project_2003370/yueyao/dataset/train-clean-100/LibriSpeech/train-clean-100

  ✓ Dataset loader imported successfully
✓ Data package initialized (circular import free)

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(

In [6]:
!python run_experiment2_simple.py \
        --data_dir "/scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2"

[2026-01-17 11:04:55] [INFO] Setting global random seed to: 42
[2026-01-17 11:04:55] [INFO] Python random seed set
[2026-01-17 11:04:55] [INFO] PYTHONHASHSEED set to: 42
[2026-01-17 11:04:55] [INFO] NumPy random seed set
[2026-01-17 11:04:55] [INFO] PyTorch CUDA seeds set
[2026-01-17 11:04:55] [INFO] PyTorch strict reproducibility mode enabled
[2026-01-17 11:04:55] [INFO] PyTorch seeds set
[2026-01-17 11:04:55] [WARNING] TensorFlow not available - skipping TensorFlow seed
[2026-01-17 11:04:55] [INFO] Environment information collected
[2026-01-17 11:04:55] [INFO] Configuring deterministic operations
[2026-01-17 11:04:55] [INFO] PyTorch deterministic operations configured
[2026-01-17 11:04:55] [INFO] Starting reproducibility verification
[2026-01-17 11:04:55] [INFO] Setting global random seed to: 42
[2026-01-17 11:04:55] [INFO] Python random seed set
[2026-01-17 11:04:55] [INFO] PYTHONHASHSEED set to: 42
[2026-01-17 11:04:55] [INFO] NumPy random seed set
[2026-01-17 11:04:55] [INFO] PyTo

In [19]:
!python run_experiment3_fewshot.py --model mel_mlp


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===

2026-01-17 11:29:51 - 